# Steam 项目 EDA

目的：按 2026-09-12 数据验收结论复核字段、主标签、时间切分和泄漏边界。

泄漏红线：`positive_ratings` / `negative_ratings` / `owners` / `average_playtime` / `median_playtime` / SteamSpy tags 与 tag votes 绝不进入 X。

In [ ]:
import pathlib
import pandas as pd
import numpy as np

# 数据约定：原始 CSV 位于 data/raw/steam_store_games/（不入库，见 data/README.md）
DATA_PATH = pathlib.Path.cwd() / "data" / "raw" / "steam_store_games" / "steam.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("缺少 data/raw/steam_store_games/steam.csv；请按 data/README.md 准备数据。")

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

## 1. 字段核对（A/B 切分）

核对下面划分是否符合实际列名：
- **主模型候选**：`release_date`, `english`, `developer`, `publisher`, `platforms`, `required_age`, `categories`, `genres`
- **快照敏感性特征**：`achievements`, `price`
- **ID**：`appid`, `name`
- **发行后结果/禁用特征**：`positive_ratings`, `negative_ratings`, `owners`, `average_playtime`, `median_playtime`, `steamspy_tags`

In [ ]:
id_cols = ['appid', 'name']
primary_feature_cols = ['release_date', 'english', 'developer', 'publisher', 'platforms',
                        'required_age', 'categories', 'genres']
snapshot_sensitivity_cols = ['achievements', 'price']
forbidden_cols = ['positive_ratings', 'negative_ratings', 'owners', 'average_playtime',
                  'median_playtime', 'steamspy_tags']

expected = id_cols + primary_feature_cols + snapshot_sensitivity_cols + forbidden_cols
print('缺失列:', [c for c in expected if c not in df.columns])
print('未归类列:', [c for c in df.columns if c not in expected])

## 2. 缺失值摸底

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])
print('\n重复 appid:', df['appid'].duplicated().sum())
print('重复 name:', df['name'].duplicated().sum())

## 3. owners 区间诊断（只作稳健性分析）

In [ ]:
df['owners'].value_counts()

In [ ]:
# 形如 "100000-200000" 取区间中点；只用于诊断，不进入主模型 X 或主标签
def owners_midpoint(s):
    lo, hi = s.split('-')
    lo, hi = int(lo), int(hi)
    return 0 if hi == 0 else (lo + hi) / 2

df['owners_mid'] = df['owners'].apply(owners_midpoint)
print(df['owners_mid'].describe())

In [ ]:
# 评价量和好评率（均为发行后结果）
total = df['positive_ratings'] + df['negative_ratings']
df['total_ratings'] = total
df['pos_rate'] = np.where(total > 0, df['positive_ratings'] / total, np.nan)
print('有评价的游戏占比:', (total > 0).mean())
print('无评价(好评率NaN)游戏数:', df['pos_rate'].isna().sum())

# 发行年份
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
print('\nrelease_year 缺失:', df['release_year'].isna().sum())
print(df['release_year'].value_counts().sort_index().to_string())

## 4. 验证已锁定的主标签

主标签：同一 `release_year` 内总评价数达到第 90 百分位。2019 年样本不完整，只用于展示偏差，不进入建模。

In [ ]:
df['hit_reviews'] = (
    df['total_ratings']
      .ge(df.groupby('release_year')['total_ratings'].transform(lambda s: s.quantile(0.90)))
      .astype('int8')
)
df['hit_owner_inclusive'] = df['owners_mid'].ge(
    df.groupby('release_year')['owners_mid'].transform(lambda s: s.quantile(0.90))
)

print('总评价数主标签，<=2018:', df.loc[df.release_year <= 2018, 'hit_reviews'].mean())
print('owners 分位数标签，全样本:', df['hit_owner_inclusive'].mean())
print('owners 分位数标签，2019:', df.loc[df.release_year == 2019, 'hit_owner_inclusive'].mean())
df.groupby('release_year')[['hit_reviews', 'hit_owner_inclusive']].mean().tail(10)

## 5. 锁定的样本与时间切分

- 建模样本：`release_year <= 2018`。
- 训练：`release_year <= 2016`。
- 验证：`release_year == 2017`。
- 最终测试：`release_year == 2018`。
- 所有预处理、编码、调参和阈值选择都只能使用对应时间点以前的数据。
- 完整决策与证据见 `docs/DECISIONS.md` 和 `docs/DATA_AUDIT.md`。